# Hito 2 - Notebook 07: Integracion de Datos (Base de Datos)
## Fase 3 de CRISP-DM - 1.4.5 Integracion de los Datos

Se integran ambos datasets preparados en una **unica base relacional** (`aldimi.db`, SQLite) mediante `db_infrastructure.py`. A partir de aqui, el modelado y el dashboard **consumen de la base de datos**, no de CSV sueltos.

In [1]:
import sys
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')
ROOT = Path.cwd()
while not (ROOT / 'src' / 'aldimi_common.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import aldimi_common as ac
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
%matplotlib inline
sns.set_theme(style='whitegrid'); plt.rcParams['figure.figsize'] = (9, 5)
pd.set_option('display.max_columns', None)
print('Raiz del proyecto:', ROOT)

Raiz del proyecto: D:\2026-01\MachLearning\finTF


In [2]:
sys.path.insert(0, str(ROOT))
import db_infrastructure as db
db_path = str(ac.DATA_PROCESSED / ac.DB_FILE)
health_csv = str(ac.DATA_PROCESSED / ac.HEALTH_PROCESSED_FILE)
stock_csv = str(ac.DATA_PROCESSED / ac.STOCK_PROCESSED_FILE)
print('Base de datos:', db_path)

Base de datos: D:\2026-01\MachLearning\finTF\data\processed\aldimi.db


### Creacion de la base e ingesta de los datasets preparados

In [3]:
db.init_db(db_path, health_csv, stock_csv)
print('Base de datos inicializada.')

Sembrando tabla 'pacientes' desde D:\2026-01\MachLearning\finTF\data\processed\Dataset_ALDIMI_Salud_Preparado.csv...


Sembrando tabla 'inventario' desde D:\2026-01\MachLearning\finTF\data\processed\Dataset_ALDIMI_Logistica_Preparado.csv...


Base de datos inicializada.


### Verificacion: lectura desde la base de datos

In [4]:
pac = db.fetch_all_pacientes(db_path)
inv = db.fetch_all_inventario(db_path)
print('Tabla pacientes :', pac.shape)
print('Tabla inventario:', inv.shape)
display(pac[['Patient_ID', 'Age', 'Prioridad_Atencion']].head())
display(inv[['Fecha', 'ID_Insumo', 'Stock_Actual', 'Alerta']].head())

Tabla pacientes : (38817, 44)
Tabla inventario: (18250, 32)


,Patient_ID,Age,Prioridad_Atencion
0,2,15,Alto
1,5,21,Bajo
2,11,24,Bajo
3,12,3,Bajo
4,13,22,Medio


,Fecha,ID_Insumo,Stock_Actual,Alerta
0,2024-01-01,SKU_1,3102,Normal
1,2024-01-02,SKU_1,3023,Normal
2,2024-01-03,SKU_1,2897,Normal
3,2024-01-04,SKU_1,2792,Normal
4,2024-01-05,SKU_1,2690,Normal


### Consulta SQL de ejemplo (auditoria/trazabilidad)

In [5]:
conn = db.get_connection(db_path)
q = '''SELECT Categoria_Insumo, COUNT(DISTINCT ID_Insumo) AS n_insumos,
              ROUND(AVG(Stock_Actual),1) AS stock_medio
       FROM inventario GROUP BY Categoria_Insumo ORDER BY stock_medio DESC;'''
resumen = pd.read_sql_query(q, conn)
conn.close()
resumen

,Categoria_Insumo,n_insumos,stock_medio
0,Medicamento Oncologico,12,2371.8
1,Alimento Especializado,13,2368.7
2,Higiene y Aseo,13,2365.3
3,Suministro Clinico,12,2323.1


### Dataset unificado (vista analitica relacional)

In [6]:
# Integracion entre frentes: la ocupacion/pacientes de alto riesgo del frente
# clinico ya se incorporo como contexto operativo en el inventario. Aqui se
# construye una vista diaria unificada que resume ambos mundos.
tasa_alto = (pac['Prioridad_Atencion'].astype(str) == 'Alto').mean()
vista = (inv.groupby('Fecha')
            .agg(Stock_Total=('Stock_Actual', 'sum'),
                 Consumo_Total=('Consumo_Diario', 'sum'),
                 Ocupacion_Total=('Ocupacion_Total', 'first'),
                 Pacientes_Alto_Riesgo=('Pacientes_Alto_Riesgo', 'first'),
                 Alertas_Criticas=('Alerta', lambda s: (s == 'Critico').sum()))
            .reset_index())
vista['Tasa_Alto_Riesgo_Global'] = round(tasa_alto, 3)
vista.to_csv(ac.DATA_PROCESSED / 'Vista_Unificada_Diaria.csv', index=False)
vista.head()

,Fecha,Stock_Total,Consumo_Total,Ocupacion_Total,Pacientes_Alto_Riesgo,Alertas_Criticas,Tasa_Alto_Riesgo_Global
0,2024-01-01,180329,5054,50,9,0,0.15
1,2024-01-02,175257,5072,53,10,0,0.15
2,2024-01-03,169979,5278,53,10,0,0.15
3,2024-01-04,164835,5144,52,9,0,0.15
4,2024-01-05,159627,5208,49,9,0,0.15


> **Conclusion / Justificacion del uso de Base de Datos:**

El uso de una base de datos (SQLite ahora; **MySQL/BigQuery** en produccion) es **obligatorio** para un proyecto de ML sostenible porque aporta:
- **Persistencia y trazabilidad**: cada paciente e insumo queda registrado y auditable.
- **Integracion**: unifica los dos frentes en un esquema comun consultable por SQL.
- **Escalabilidad**: soporta el crecimiento de 50 a 100 familias y el historial creciente.
- **Desacople**: los modelos y el dashboard **consumen de la BD** y no de archivos CSV sueltos, habilitando reentrenamiento y monitoreo continuos.